# Property Price Prediction — Model Training & Evaluation

Step 1: load `bronze_listings.csv` and clean it, mirroring the rules already proven in `backend/scripts/ingest-bronze-listings.ts` (`normalizePropertyType`, `parsePrice`, `parseSoldDate`) so a row considered "valid" here matches what the production database accepts. Also parses `sold_year` and `sold_month` from `listing_date` — training features capturing price growth/seasonality over time (the live prediction endpoint will fill in the current year/month instead, since it's predicting a present-day value, not a past sale).

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

DATA_AI_DIR = Path.cwd().parent

listings = pd.read_csv(DATA_AI_DIR / "bronze_listings.csv", dtype=str)
print(f"Raw rows: {len(listings):,}")
listings.head(3)

Raw rows: 42,320


,listing_id,source,address,suburb,postcode,state,property_type,bedrooms,bathrooms,parking,land_size_sqm,price,listing_date,listing_description,raw_payload,scraped_at
0,14465,domain_com_au,56 Selwyn Street,Paddington,NaN,NSW,House,4,2,2,203,"$3,450,000",Sold at auction 24 Jul 2026,NaN,"{""price"": ""$3,450,000"", ""suburb"": ""Paddington""...",2026-07-25 18:33:21.551505
1,14466,domain_com_au,11D/4 Distillery Drive,Pyrmont,NaN,NSW,Apartment / Unit / Flat,3,3,2,210,"$3,500,000",Sold by private treaty 16 Jul 2026,NaN,"{""price"": ""$3,500,000"", ""suburb"": ""Pyrmont"", ""...",2026-07-25 18:33:21.555643
2,14467,domain_com_au,2/8 Telopea Avenue,Caringbah South,NaN,NSW,NaN,3,2,2,200,"$1,525,000",Sold by private treaty 13 Jul 2026,NaN,"{""price"": ""$1,525,000"", ""suburb"": ""Caringbah S...",2026-07-25 18:33:21.556642


In [2]:
PROPERTY_TYPE_MAP = [
    (re.compile("townhouse", re.I), "Townhouse"),
    (re.compile("villa", re.I), "Villa"),
    (re.compile("apartment", re.I), "Apartment"),
    (re.compile("unit|flat", re.I), "Unit"),
    (re.compile("house", re.I), "House"),
]


def normalize_property_type(raw):
    if not isinstance(raw, str) or not raw.strip():
        return None
    for pattern, label in PROPERTY_TYPE_MAP:
        if pattern.search(raw):
            return label
    return None


def parse_price(raw):
    if not isinstance(raw, str):
        return None
    cleaned = re.sub(r"[^0-9.]", "", raw)
    if not cleaned:
        return None
    value = float(cleaned)
    return value if value > 0 else None


MONTH_NUMBERS = {
    "jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
    "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12,
}

SOLD_DATE_RE = re.compile(
    r"(\d{1,2})\s+(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+(\d{4})", re.I
)


def parse_sold_year(raw):
    if not isinstance(raw, str):
        return None
    match = SOLD_DATE_RE.search(raw)
    return int(match.group(3)) if match else None


def parse_sold_month(raw):
    if not isinstance(raw, str):
        return None
    match = SOLD_DATE_RE.search(raw)
    return MONTH_NUMBERS[match.group(2)[:3].lower()] if match else None


listings["property_type_clean"] = listings["property_type"].apply(normalize_property_type)
listings["price_clean"] = listings["price"].apply(parse_price)
listings["sold_year"] = listings["listing_date"].apply(parse_sold_year)
listings["sold_month"] = listings["listing_date"].apply(parse_sold_month)
for col in ["bedrooms", "bathrooms", "parking", "land_size_sqm"]:
    listings[col] = pd.to_numeric(listings[col], errors="coerce")

listings[["property_type", "property_type_clean", "price", "price_clean", "listing_date", "sold_year", "sold_month"]].head(5)

,property_type,property_type_clean,price,price_clean,listing_date,sold_year,sold_month
0,House,House,"$3,450,000",3450000.0,Sold at auction 24 Jul 2026,2026.0,7.0
1,Apartment / Unit / Flat,Apartment,"$3,500,000",3500000.0,Sold by private treaty 16 Jul 2026,2026.0,7.0
2,NaN,None,"$1,525,000",1525000.0,Sold by private treaty 13 Jul 2026,2026.0,7.0
3,NaN,None,"$740,000",740000.0,Sold by private treaty 06 Jul 2026,2026.0,7.0
4,House,House,"$2,250,000",2250000.0,Sold prior to auction 03 Jul 2026,2026.0,7.0


`postcode` is empty for every row in `bronze_listings.csv` — confirmed directly (`df["postcode"].notna().sum() == 0`), not just assumed. The production ingestion script doesn't get it from the CSV either: it looks up suburb → postcode from an external Australian postcode dataset (first match wins). Replicating that same lookup here, so the notebook's postcode values match what the production database actually has.

In [3]:
POSTCODE_LOOKUP_URL = "https://raw.githubusercontent.com/Elkfox/Australian-Postcode-Data/master/au_postcodes.csv"

postcode_lookup_raw = pd.read_csv(POSTCODE_LOOKUP_URL, dtype=str)
postcode_lookup_raw["place_name"] = postcode_lookup_raw["place_name"].str.strip().str.lower()
postcode_lookup_raw["state_code"] = postcode_lookup_raw["state_code"].str.strip().str.upper()

# First match wins, same rule as ingest-bronze-listings.ts's loadPostcodeLookup.
postcode_lookup = (
    postcode_lookup_raw.dropna(subset=["place_name", "state_code", "postcode"])
    .drop_duplicates(subset=["place_name", "state_code"], keep="first")
    .set_index(["place_name", "state_code"])["postcode"]
)
print(f"Loaded {len(postcode_lookup):,} suburb->postcode mappings.")

lookup_key = pd.MultiIndex.from_arrays(
    [listings["suburb"].str.strip().str.lower(), listings["state"].str.strip().str.upper()]
)
listings["postcode"] = postcode_lookup.reindex(lookup_key).values

matched_postcode = listings["postcode"].notna().sum()
print(f"Rows with a resolved postcode: {matched_postcode:,} / {len(listings):,} ({matched_postcode / len(listings):.1%})")

Loaded 16,013 suburb->postcode mappings.
Rows with a resolved postcode: 41,962 / 42,320 (99.2%)


In [4]:
clean = listings.dropna(
    subset=[
        "property_type_clean", "price_clean", "bedrooms", "bathrooms",
        "land_size_sqm", "suburb", "state", "postcode", "sold_year", "sold_month",
    ]
).copy()
# price >= $10,000: drops nominal/non-market transactions (found: $1, $1,000,
# $2,000 rows — family transfers or data-entry errors, confirmed by manual
# inspection, not real sales) while keeping legitimate low-value regional
# sales (e.g. $45,000 houses in Cobar, NSW).
clean = clean[(clean["price_clean"] >= 10_000) & (clean["land_size_sqm"] > 0)]

print(f"Raw rows:   {len(listings):,}")
print(f"Clean rows: {len(clean):,}  (dropped {len(listings) - len(clean):,})")
clean[["suburb", "state", "postcode", "property_type_clean", "bedrooms", "bathrooms", "parking", "land_size_sqm", "sold_year", "sold_month", "price_clean"]].describe(include="all").T

Raw rows:   42,320
Clean rows: 32,541  (dropped 9,779)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
suburb,32541,1429,Orange,368,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state,32541,1,NSW,32541,NaN,NaN,NaN,NaN,NaN,NaN,NaN
postcode,32541,547,2765,664,NaN,NaN,NaN,NaN,NaN,NaN,NaN
property_type_clean,32541,3,House,31266,NaN,NaN,NaN,NaN,NaN,NaN,NaN
bedrooms,32541.0,NaN,NaN,NaN,3.625242,0.938699,1.0,3.0,4.0,4.0,16.0
bathrooms,32541.0,NaN,NaN,NaN,1.929597,0.80643,1.0,1.0,2.0,2.0,14.0
parking,30726.0,NaN,NaN,NaN,1.977055,0.969525,1.0,1.0,2.0,2.0,20.0
land_size_sqm,32541.0,NaN,NaN,NaN,521.489383,210.725896,200.0,347.0,480.0,680.0,999.0
sold_year,32541.0,NaN,NaN,NaN,2025.311791,0.696743,2023.0,2025.0,2025.0,2026.0,2026.0
sold_month,32541.0,NaN,NaN,NaN,6.408469,3.478049,1.0,3.0,6.0,10.0,12.0


## 2. Enrich with suburb-level market context

`domain_suburb_insights.csv` has real market stats (median sold price, rental yield, auction clearance, days on market) per `postcode` + `property_category` + `bedrooms` bucket. Aggregated per `postcode` + `property_category` (averaged across bedroom buckets) so it can join onto each listing via `postcode` — the key relationship we confirmed earlier (postcode maps to these stats, not a raw model feature itself).

In [5]:
suburb_insights = pd.read_csv(DATA_AI_DIR / "domain_suburb_insights.csv", dtype={"postcode": str})
print("property_category values in this file:", suburb_insights["property_category"].unique())

# 0 in these columns means "no reliable data for this bedroom bucket" (see
# the diagnostic above), not a real value — treat as missing so .mean()
# skips it instead of dragging the aggregate toward zero. rental_yield_pct
# doesn't need this; it has no zero values.
ZERO_AS_MISSING_COLS = ["median_sold_price", "days_on_market", "auction_clearance_rate"]
for col in ZERO_AS_MISSING_COLS:
    suburb_insights[col] = suburb_insights[col].replace(0, np.nan)

suburb_agg = (
    suburb_insights.groupby(["postcode", "property_category"])
    .agg(
        suburb_median_price=("median_sold_price", "mean"),
        suburb_days_on_market=("days_on_market", "mean"),
        suburb_auction_clearance_pct=("auction_clearance_rate", "mean"),
        suburb_rental_yield_pct=("rental_yield_pct", "mean"),
    )
    .reset_index()
)
suburb_agg.head()

property_category values in this file: ['House' 'Unit']


,postcode,property_category,suburb_median_price,suburb_days_on_market,suburb_auction_clearance_pct,suburb_rental_yield_pct
0,2000,Unit,4.493533e+06,133.000000,NaN,3.433333
1,2007,House,NaN,NaN,NaN,NaN
2,2007,Unit,1.053667e+06,57.000000,0.545455,5.723333
3,2009,House,NaN,NaN,NaN,NaN
4,2009,Unit,1.765000e+06,71.666667,0.451073,4.006667


In [6]:
# bronze's 5-way property_type_clean doesn't match domain's 2-category
# vocabulary (House / Unit only) — map onto the closest equivalent for the join.
CATEGORY_MAP = {
    "House": "House",
    "Townhouse": "House",
    "Villa": "House",
    "Apartment": "Unit",
    "Unit": "Unit",
}
clean["property_category_for_join"] = clean["property_type_clean"].map(CATEGORY_MAP)

merged = clean.merge(
    suburb_agg,
    left_on=["postcode", "property_category_for_join"],
    right_on=["postcode", "property_category"],
    how="left",
)

matched = merged["suburb_median_price"].notna().sum()
print(f"Rows with suburb market data matched: {matched:,} / {len(merged):,} ({matched / len(merged):.1%})")

market_cols = ["suburb_median_price", "suburb_days_on_market", "suburb_auction_clearance_pct", "suburb_rental_yield_pct"]
# NOTE: unmatched rows are NOT filled here — that would require a fallback
# statistic (a median), and any such statistic must be fit on the training
# set only, after the train/test split below, or it leaks test-set
# information into values used at training time.
merged[["suburb", "postcode", "property_type_clean"] + market_cols].head()

Rows with suburb market data matched: 29,821 / 32,541 (91.6%)


,suburb,postcode,property_type_clean,suburb_median_price,suburb_days_on_market,suburb_auction_clearance_pct,suburb_rental_yield_pct
0,Paddington,2021,House,3822500.0,58.000000,0.608273,2.376667
1,Pyrmont,2009,Apartment,1765000.0,71.666667,0.451073,4.006667
2,Alexandria,1435,House,NaN,NaN,NaN,NaN
3,Thornton,2322,House,873375.0,25.000000,NaN,4.265556
4,Manly,1655,Apartment,NaN,NaN,NaN,NaN


In [7]:
# Check how many rows in the REAL merged training data are affected by the
# zero-as-missing pattern found in domain_suburb_insights.csv, and whether
# the other three stat columns show the same issue.
for col in market_cols:
    zero_count = (merged[col] == 0).sum()
    print(f"{col}: {zero_count:,} / {len(merged):,} rows are exactly 0 ({zero_count / len(merged):.1%})")

print()
print("Same check on the raw (pre-aggregation) domain_suburb_insights.csv:")
for raw_col in ["median_sold_price", "days_on_market", "auction_clearance_rate", "rental_yield_pct"]:
    zero_count = (suburb_insights[raw_col] == 0).sum()
    print(f"{raw_col}: {zero_count:,} / {len(suburb_insights):,} rows are exactly 0 ({zero_count / len(suburb_insights):.1%})")

suburb_median_price: 0 / 32,541 rows are exactly 0 (0.0%)
suburb_days_on_market: 0 / 32,541 rows are exactly 0 (0.0%)
suburb_auction_clearance_pct: 0 / 32,541 rows are exactly 0 (0.0%)
suburb_rental_yield_pct: 0 / 32,541 rows are exactly 0 (0.0%)

Same check on the raw (pre-aggregation) domain_suburb_insights.csv:
median_sold_price: 0 / 7,747 rows are exactly 0 (0.0%)
days_on_market: 0 / 7,747 rows are exactly 0 (0.0%)
auction_clearance_rate: 0 / 7,747 rows are exactly 0 (0.0%)
rental_yield_pct: 0 / 7,747 rows are exactly 0 (0.0%)


## 3. Train/test split — time-based, not random

The real deployment scenario is predicting a *current* price from *past* sales — a random split would let the model train on later sales and get tested on earlier ones, which doesn't reflect that. Sorting by `sold_year`/`sold_month` and taking the earliest ~80% as train, most recent ~20% as test, avoids that look-ahead bias. This must happen **before** fitting any imputation statistic (the `parking` and suburb-fallback medians) — otherwise those statistics leak test-set information into training, which is exactly the bug found in the previous version of this notebook.

In [8]:
merged_sorted = merged.sort_values(["sold_year", "sold_month"], kind="stable").reset_index(drop=True)

split_idx = int(len(merged_sorted) * 0.8)
train_df = merged_sorted.iloc[:split_idx].copy()
test_df = merged_sorted.iloc[split_idx:].copy()

last_train = train_df[["sold_year", "sold_month"]].iloc[-1]
first_test = test_df[["sold_year", "sold_month"]].iloc[0]
print(f"Train: {len(train_df):,} rows, last row sold {last_train['sold_year']:.0f}-{last_train['sold_month']:.0f}")
print(f"Test:  {len(test_df):,} rows, first row sold {first_test['sold_year']:.0f}-{first_test['sold_month']:.0f}")

Train: 26,032 rows, last row sold 2026-4
Test:  6,509 rows, first row sold 2026-4


`parking` is missing for 5.6% of rows. Checked whether this correlates with property type (which would suggest missing = genuinely 0, common for apartments) — it doesn't: House and Apartment/Unit have similar missing rates (~5.4% and ~6.0%), so this looks like a general scraping gap, not a hidden zero. Imputing with the median **per `property_type_clean`, fit on `train_df` only** — the same fitted values are then applied to `test_df`, so no test-set information leaks into the imputation.

In [9]:
before_train = train_df["parking"].isna().sum()
before_test = test_df["parking"].isna().sum()

# Fit per-type median on train only.
parking_medians = train_df.groupby("property_type_clean")["parking"].median()

train_df["parking"] = train_df.apply(
    lambda row: parking_medians[row["property_type_clean"]] if pd.isna(row["parking"]) else row["parking"], axis=1
)
test_df["parking"] = test_df.apply(
    lambda row: parking_medians[row["property_type_clean"]] if pd.isna(row["parking"]) else row["parking"], axis=1
)

print(f"train parking missing: {before_train:,} -> {train_df['parking'].isna().sum():,}")
print(f"test  parking missing: {before_test:,} -> {test_df['parking'].isna().sum():,}")

train parking missing: 1,591 -> 0
test  parking missing: 224 -> 0


Rows with no suburb-market match (8.4%, from step 2) still need their `suburb_*` columns filled — same rule as `parking`: fit the fallback median on `train_df` only, apply the same fitted values to `test_df`.

In [10]:
for col in market_cols:
    fallback_value = train_df[col].median()
    train_df[col] = train_df[col].fillna(fallback_value)
    test_df[col] = test_df[col].fillna(fallback_value)

print("Remaining nulls after fallback fill:")
print(pd.concat([train_df[market_cols].isna().sum(), test_df[market_cols].isna().sum()], axis=1, keys=["train", "test"]))

Remaining nulls after fallback fill:
                              train  test
suburb_median_price               0     0
suburb_days_on_market             0     0
suburb_auction_clearance_pct      0     0
suburb_rental_yield_pct           0     0


## 4. Encode `property_type_clean`

The only categorical feature — one-hot encoded (5 values: House, Townhouse, Villa, Apartment, Unit). `suburb`/`state`/`postcode` are join keys only, not model inputs. Encoder is **fit on `train_df` only** (`handle_unknown="ignore"` in case `test_df` has a category `train_df` doesn't), same discipline as the imputation steps above.

In [11]:
from sklearn.preprocessing import OneHotEncoder

print("property_type_clean values in train:", sorted(train_df["property_type_clean"].unique()))
print("property_type_clean values in test: ", sorted(test_df["property_type_clean"].unique()))

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(train_df[["property_type_clean"]])

train_encoded = pd.DataFrame(
    encoder.transform(train_df[["property_type_clean"]]),
    columns=encoder.get_feature_names_out(["property_type_clean"]),
    index=train_df.index,
)
test_encoded = pd.DataFrame(
    encoder.transform(test_df[["property_type_clean"]]),
    columns=encoder.get_feature_names_out(["property_type_clean"]),
    index=test_df.index,
)

train_encoded.head()

property_type_clean values in train: ['Apartment', 'House', 'Unit']
property_type_clean values in test:  ['Apartment', 'House']


,property_type_clean_Apartment,property_type_clean_House,property_type_clean_Unit
0,0.0,1.0,0.0
1,0.0,1.0,0.0
2,0.0,1.0,0.0
3,0.0,1.0,0.0
4,1.0,0.0,0.0


## 5. Assemble final feature matrices

Numeric features (`bedrooms`, `bathrooms`, `parking`, `land_size_sqm`, `sold_year`, `sold_month`, the 4 `suburb_*` columns) concatenated with the one-hot encoded columns from step 4. Target is `log1p(price_clean)` — real-estate prices are right-skewed (confirmed earlier: min $1, max $26.1M, mean $1.55M vs. median $1.2M), so training on the log scale keeps the loss from being dominated by the highest-value properties. Predictions are converted back with `expm1` at evaluation time.

In [12]:
NUMERIC_FEATURES = ["bedrooms", "bathrooms", "parking", "land_size_sqm", "sold_year", "sold_month"] + market_cols

X_train = pd.concat([train_df[NUMERIC_FEATURES].reset_index(drop=True), train_encoded.reset_index(drop=True)], axis=1)
X_test = pd.concat([test_df[NUMERIC_FEATURES].reset_index(drop=True), test_encoded.reset_index(drop=True)], axis=1)

y_train = np.log1p(train_df["price_clean"]).reset_index(drop=True)
y_test = np.log1p(test_df["price_clean"]).reset_index(drop=True)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
X_train.head()

X_train: (26032, 13), X_test: (6509, 13)


,bedrooms,bathrooms,parking,land_size_sqm,sold_year,sold_month,suburb_median_price,suburb_days_on_market,suburb_auction_clearance_pct,suburb_rental_yield_pct,property_type_clean_Apartment,property_type_clean_House,property_type_clean_Unit
0,2.0,1.0,2.0,215.0,2023.0,2.0,2.412500e+06,43.500000,0.608856,2.296667,0.0,1.0,0.0
1,2.0,1.0,2.0,215.0,2023.0,2.0,2.412500e+06,43.500000,0.608856,2.296667,0.0,1.0,0.0
2,3.0,2.0,1.0,215.0,2023.0,2.0,3.062083e+06,48.500000,0.715015,2.478000,0.0,1.0,0.0
3,3.0,1.0,2.0,220.0,2023.0,2.0,1.313333e+06,26.333333,0.641304,3.365000,0.0,1.0,0.0
4,2.0,2.0,2.0,219.0,2023.0,3.0,7.146667e+05,45.666667,0.321429,5.370000,1.0,0.0,0.0


## 6. Baseline models

Two baselines, evaluated on `X_test`/`y_test` in real dollar terms (predictions converted back from log scale with `expm1`):

- **Naive median** — always predicts `y_train`'s median price. The true floor: any real model must beat this or the features aren't adding value.
- **Linear Regression** — fit on `log1p(price)`, interpretable and fast. Expected to underfit nonlinear location/size interactions compared to tree-based models added later, but gives a fair first comparison point since the log transform already addresses price skew.

In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def evaluate(name, y_true_log, y_pred_log):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    print(f"{name:20s}  MAE: ${mae:,.0f}   RMSE: ${rmse:,.0f}   R2: {r2:.3f}")
    return {"name": name, "mae": mae, "rmse": rmse, "r2": r2}


# Naive median baseline: constant prediction, ignores all features.
median_pred_log = np.full_like(y_test, y_train.median())
median_result = evaluate("Naive median", y_test, median_pred_log)

# Linear regression baseline.
linreg = LinearRegression()
linreg.fit(X_train, y_train)
linreg_pred_log = linreg.predict(X_test)
linreg_result = evaluate("Linear Regression", y_test, linreg_pred_log)

Naive median          MAE: $529,587   RMSE: $901,066   R2: -0.016
Linear Regression     MAE: $286,687   RMSE: $611,136   R2: 0.532


## 7. Candidate main models — RandomForest vs. GradientBoosting

Both are tree-based ensembles that can capture nonlinear location/size interactions the linear baseline can't. Running both (cheap at this data size) rather than picking one on intuition — whichever wins on test MAE/R² becomes the model actually shipped in the FastAPI service. Default hyperparameters first; only worth tuning the winner further.

In [14]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor

rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred_log = rf.predict(X_test)
rf_result = evaluate("RandomForest", y_test, rf_pred_log)

gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train, y_train)
gb_pred_log = gb.predict(X_test)
gb_result = evaluate("GradientBoosting", y_test, gb_pred_log)

RandomForest          MAE: $231,982   RMSE: $500,949   R2: 0.686


GradientBoosting      MAE: $238,275   RMSE: $503,324   R2: 0.683


## 8. Hyperparameter tuning — RandomForest

Winner from step 7. Testing 3 values of `n_estimators` x 2 values of `max_depth` (10, None) — 6 combinations total, all other params left at default.

In [15]:
tuning_results = []
for n_estimators in [200, 400, 600]:
    for max_depth in [10, None]:
        model = RandomForestRegressor(
            n_estimators=n_estimators, max_depth=max_depth, random_state=42, n_jobs=-1
        )
        model.fit(X_train, y_train)
        pred_log = model.predict(X_test)
        result = evaluate(f"RF n_estimators={n_estimators}, max_depth={max_depth}", y_test, pred_log)
        result["n_estimators"] = n_estimators
        result["max_depth"] = max_depth
        result["model"] = model
        tuning_results.append(result)

best = min(tuning_results, key=lambda r: r["mae"])
print(f"\nBest: n_estimators={best['n_estimators']}, max_depth={best['max_depth']} (MAE ${best['mae']:,.0f})")

RF n_estimators=200, max_depth=10  MAE: $230,421   RMSE: $494,191   R2: 0.694


RF n_estimators=200, max_depth=None  MAE: $231,909   RMSE: $500,674   R2: 0.686


RF n_estimators=400, max_depth=10  MAE: $230,533   RMSE: $494,615   R2: 0.694


RF n_estimators=400, max_depth=None  MAE: $231,981   RMSE: $500,576   R2: 0.686


RF n_estimators=600, max_depth=10  MAE: $230,545   RMSE: $494,636   R2: 0.694


RF n_estimators=600, max_depth=None  MAE: $231,933   RMSE: $500,375   R2: 0.687

Best: n_estimators=200, max_depth=10 (MAE $230,421)
